In [ ]:
# Imports
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

In [ ]:
# Fill in your file path here
df = pd.read_csv('')
df = df.drop(columns = ['Unnamed: 0'])
df.columns

In [ ]:
# Preview training data
train = df[df['season'] <= 2024]
train

In [ ]:
# Get VIF values
X = df[['age', 'gs', 'cmp_pct', 'td_pct', 'int_pct', 'one_d', 'succ_pct', 'y_a', 'ay_a', 
        'y_c', 'y_g', 'qbr', 'sk_pct', 'four_qc', 'gwd', 'Wins', 'tot_TD', 'tot_Yds', 'cpoe',
        'rtg', 'aggressiveness', 'db_epa', 'TO']]

y = df['MVP']

X = sm.add_constant(X)  
X = X.astype(float)

vif = pd.DataFrame()
vif['feature'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif)

In [ ]:
# Whats better, a model that has a high psuedo R-squared or one thats right?
# Choose up to 5 features
X = train[[]]
y = train['MVP']
X = X.astype(float)

X = sm.add_constant(X)
model = sm.Logit(y, X).fit()
print(model.summary())

In [ ]:
# Get data we are testing on
test = df[df['season'] == 2025]

test = test[['player', 'age', 'gs', 'cmp_pct', 'td_pct', 'int_pct', 'y_g', 'qbr', 'sk_pct', 'succ_pct', 'four_qc', 'gwd', 
             'Wins', 'tot_TD', 'tot_Yds', 'cpoe', 'rtg', 'aggressiveness', 'db_epa', 'TO']]

In [ ]:
# Get predictions and make small chart
test = test[['player','aggressiveness','cpoe', 'rtg','age','Wins', 'tot_TD','succ_pct','gwd','qbr','TO','db_epa', 'sk_pct']]
features = [] # Copy the features you chose into here as well

df.columns = df.columns.astype(str)
test.columns = test.columns.astype(str)
train.columns = train.columns.astype(str)

X = train[features]
y = train['MVP']

X_CT = test[features]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

test['MVP_Prob'] = model.predict_proba(X_CT)[:, 1]
top_candidates = test[['player', 'MVP_Prob']].sort_values('MVP_Prob', ascending=False)
print(top_candidates.head(10))

top_candidates = top_candidates.head(5)
plt.bar(top_candidates['player'], top_candidates['MVP_Prob'])
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# Calculate ROC AUC score, and make plot
y_score = model.predict_proba(X)[:, 1] 

auc = roc_auc_score(y, y_score)
print(f"ROC AUC Score: {auc:.4f}")

fpr, tpr, thresholds = roc_curve(y, y_score)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', label=f'ROC curve (AUC = {auc:.2f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()

# STOP HERE!

In [ ]:
# Why does it sometimes give us an error when predicting past years?
check = df[df['season'] <= 2019]
colors = check['MVP'].map({1: 'maroon', 0: 'steelblue'})
plt.scatter(check['Wins'], check['tot_TD'], c=colors)
plt.xlabel('Wins')
plt.ylabel('tot_TD')
plt.show()

In [ ]:
# How can we explain why the model gave us a certain result? (need to adjust train timeframe)
df['23lj'] = ((df['season'] == 2023) & (df['player'] == 'Lamar Jackson')).astype(int)

In [ ]:
test_2023 = df[df['season'] == 2023].copy()

X_2023 = test_2023[features]
test_2023['MVP_Prob'] = model.predict_proba(X_2023)[:, 1]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, feat in zip(axes, features):
    colors = test_2023['23lj'].map({1: 'maroon', 0: 'steelblue'})
    ax.scatter(test_2023[feat], test_2023['MVP_Prob'], c=colors)
    ax.set_xlabel(feat)
    ax.set_ylabel('MVP_Prob')

plt.tight_layout()
plt.show()